# Task 12A–B: Documented Inpatient Palliative-Care Use

This notebook identifies normalized ICD-10-CM `Z51.5` (`Z515`) in any diagnosis position among adult hematologic-malignancy hospitalizations. The variable is described as **documented inpatient palliative-care use**, not palliative-care consultation. Run all cells to refresh the results.

## Methods

Diagnosis codes were converted to uppercase and stripped of decimal points and spaces during the Phase 1–2 cohort build. `palliative_care=1` when `Z515` occurs in any diagnosis field. Counts and prevalence estimates use `DISCWT`. Confidence intervals, the difference test, and the crude weighted odds-ratio inference use year-specific `NIS_STRATUM` strata with Taylor linearization. Because `HOSP_NIS` is unavailable by study decision, sampled discharges—not hospitals—are treated as variance units; these are strata-adjusted approximations, not full NIS survey-design estimates.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.phase_6_palliative_care import main
summary = main()

{
  "definition": "Documented inpatient palliative-care use: normalized Z51.5 (Z515) in any diagnosis position.",
  "normalization": "Uppercase; decimal points and spaces removed in the Phase 1\u20132 cohort build.",
  "records_excluded_for_missing_weight_or_stratum": 0,
  "normalization_validation": {
    "flagged_records": 80733,
    "records_containing_normalized_Z515": 80733,
    "discordant_records": 0
  },
  "variance_note": "NIS_STRATUM and year-specific strata used in Taylor linearization; sampled discharges are variance units because HOSP_NIS is unavailable by study decision. Not a full NIS design variance estimate.",
  "prevalence_table": [
    {
      "cohort": "All adult HM hospitalizations",
      "unweighted_sample_n": 994992,
      "unweighted_palliative_care_n": 80733,
      "weighted_hospitalizations": 4974958,
      "weighted_palliative_care_n": 403665,
      "weighted_palliative_care_percent": 8.11,
      "ci_95_lower_percent": 8.06,
      "ci_95_upper_percent": 8.17

## Weighted prevalence

In [2]:
prevalence = pd.read_csv(REPO_ROOT / 'outputs/phase_6/palliative_care_prevalence.csv')
prevalence.columns = [
    'Cohort', 'Unweighted sample n', 'Unweighted palliative-care n',
    'Weighted hospitalizations', 'Weighted palliative-care n',
    'Weighted palliative-care %', '95% CI lower, %', '95% CI upper, %'
]
display(prevalence.style.hide(axis='index').format(thousands=','))

Cohort,Unweighted sample n,Unweighted palliative-care n,Weighted hospitalizations,Weighted palliative-care n,Weighted palliative-care %,"95% CI lower, %","95% CI upper, %"
All adult HM hospitalizations,"994,992","80,733","4,974,958","403,665",8.110000,8.060000,8.170000
HM without documented sepsis,"836,571","54,772","4,182,853","273,860",6.550000,6.490000,6.600000
HM with documented sepsis,"158,421","25,961","792,105","129,805",16.390000,16.210000,16.570000


## Sepsis comparison

In [3]:
comparison = pd.DataFrame([
    {'Measure': 'Absolute prevalence difference, percentage points', 'Estimate': summary['comparison']['absolute_difference_percentage_points'], '95% CI': f"{summary['comparison']['difference_ci_95_lower']:.2f} to {summary['comparison']['difference_ci_95_upper']:.2f}", 'P-value': summary['comparison']['strata_adjusted_p_value']},
    {'Measure': 'Crude weighted odds ratio: sepsis vs no sepsis', 'Estimate': summary['comparison']['crude_weighted_odds_ratio'], '95% CI': f"{summary['comparison']['odds_ratio_ci_95_lower']:.3f} to {summary['comparison']['odds_ratio_ci_95_upper']:.3f}", 'P-value': summary['comparison']['odds_ratio_p_value']},
])
display(comparison.style.hide(axis='index'))

Measure,Estimate,95% CI,P-value
"Absolute prevalence difference, percentage points",9.840000,9.65 to 10.03,<0.001
Crude weighted odds ratio: sepsis vs no sepsis,2.798000,2.753 to 2.842,<0.001


## Phenotype validation

In [4]:
validation = pd.DataFrame([summary['normalization_validation']]).rename(columns={
    'flagged_records': 'Flagged palliative-care records',
    'records_containing_normalized_Z515': 'Records containing normalized Z515',
    'discordant_records': 'Discordant records',
})
display(validation.style.hide(axis='index').format(thousands=','))

Flagged palliative-care records,Records containing normalized Z515,Discordant records
"80,733","80,733",0


## Copy/paste-friendly Markdown

The next cell prints both result tables as plain Markdown.

In [5]:
def print_markdown(dataframe, title):
    print(f'## {title}\n')
    headers = list(dataframe.columns)
    print('| ' + ' | '.join(headers) + ' |')
    print('|' + '|'.join(['---'] * len(headers)) + '|')
    for row in dataframe.astype(str).itertuples(index=False, name=None):
        print('| ' + ' | '.join(value.replace('|', '\\|') for value in row) + ' |')
    print()
print_markdown(prevalence, 'Documented inpatient palliative-care use')
print_markdown(comparison, 'Unadjusted sepsis comparison')

## Documented inpatient palliative-care use

| Cohort | Unweighted sample n | Unweighted palliative-care n | Weighted hospitalizations | Weighted palliative-care n | Weighted palliative-care % | 95% CI lower, % | 95% CI upper, % |
|---|---|---|---|---|---|---|---|
| All adult HM hospitalizations | 994992 | 80733 | 4974958 | 403665 | 8.11 | 8.06 | 8.17 |
| HM without documented sepsis | 836571 | 54772 | 4182853 | 273860 | 6.55 | 6.49 | 6.6 |
| HM with documented sepsis | 158421 | 25961 | 792105 | 129805 | 16.39 | 16.21 | 16.57 |

## Unadjusted sepsis comparison

| Measure | Estimate | 95% CI | P-value |
|---|---|---|---|
| Absolute prevalence difference, percentage points | 9.84 | 9.65 to 10.03 | <0.001 |
| Crude weighted odds ratio: sepsis vs no sepsis | 2.798 | 2.753 to 2.842 | <0.001 |



## Interpretation

Documented inpatient palliative-care use was more common among HM hospitalizations with documented sepsis than among those without documented sepsis. This is an unadjusted association and should not be interpreted causally. Covariate-adjusted probabilities belong to the later multivariable analysis in Command 16B and are not estimated in Task 12B.